<a href="https://colab.research.google.com/github/zamanuddinkhan/Python-AI-LLM/blob/main/Demo5_Memory_Checkpointing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain==0.3.27 langchain-core==0.3.72 langchain-openai==0.3.28 langchain-text-splitters==0.3.9 langgraph==0.6.1

In [ ]:
!pip install tavily-python langchain_community

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## Single LLM Agent

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Begin by creating a StateGraph. A StateGraph object outlines the structure of the chatbot as a "state machine."
# We'll incorporate nodes to represent the LLM and the functions that the agent can invoke,
# while edges will define how the agent transitions between these functions.

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

# Each node can receive the current State as input and output an update to the state.
# Updates to messages will be appended to the existing list rather than overwriting it,
# thanks to the prebuilt add_messages function used with the Annotated syntax.

graph_builder = StateGraph(State)

In [ ]:
# Load the LLM
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

# Create LLM invoke function
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

# Creating first node

graph_builder.add_node("chatbot", chatbot)

# The first argument is the unique node name
# The second argument is the function or object that will be called whenever the node is used.

Observe that the node receives the current State as input and returns a dictionary with an updated "messages" list. This is the standard pattern for all LangGraph node functions.

The add_messages function in our State appends the LLM's response messages to the existing messages in the state.









In [ ]:
# creating edges

# Add an entry point to specify where the graph should begin its process each time it runs.
# This defines the starting point of the workflow within the graph.
graph_builder.add_edge(START, "chatbot")
# Similarly, define a finish point to indicate that "whenever this node is executed, the process can terminate."
# This serves as the endpoint for the graph's workflow.
graph_builder.add_edge("chatbot", END)

In [ ]:
# Finally, to execute the graph, call the compile() method on the graph builder.
# This will prepare the graph for execution.
# This generates a CompiledGraph that can be invoked on our state, allowing us to execute the graph's defined workflow.
graph = graph_builder.compile()

In [ ]:
# Display the Graph
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
# Passing a input to test the Agent

user_input = "What do you know about AI?"
print("User: " + user_input)

for event in graph.stream({"messages": [("user", user_input)]}):
  for value in event.values():
    print("Assistant:", value["messages"][-1].content)

## Adding tools to this Simple Agent

We will using Tavily search engine. Get tavily API key from https://tavily.com/

In [ ]:
from google.colab import userdata
import os
os.environ['TAVILY_API_KEY']=userdata.get('TAVILY_API_KEY')

In [ ]:
# tavily is available from langchain community library
from langchain_community.tools.tavily_search import TavilySearchResults

tool = TavilySearchResults(max_results=5)
tools = [tool]

# testing
tool.invoke("What is ML?")

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

# ToolNode -> A node that runs the tools called in the last AIMessage.
# It can be used either in StateGraph with a "messages" state key (or a custom key passed via ToolNode's 'messages_key').
# If multiple tool calls are requested, they will be run in parallel.
# The output will be a list of ToolMessages, one for each tool call.


# tools_condition -> Use in the conditional_edge to route to the ToolNode if the last message has tool calls.
# Otherwise, route to the end.

In [ ]:
# New Agent

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

tool = TavilySearchResults(max_results=2)
tools = [tool]

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile()

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
# Passing a input to test the Agent with tool calling

user_input = "what is latest news about AI?"
print("User: " + user_input)

for event in graph.stream({"messages": [("user", user_input)]}):
  for value in event.values():
    print("Assistant:", value["messages"][-1].content)

## Add memory to the Agent

As discussed earlier, memory can be very useful in maintaining the context for the LLM and provide relevant answers to next user queries.

LangGraph has persistent checkpointing. By providing a checkpointer during the graph compilation and specifying a thread_id when invoking the graph, LangGraph automatically saves the state after each step. When the graph is invoked again using the same thread_id, it loads the saved state, enabling the chatbot to resume from where it previously left off.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

Currently using an memory checkpointer. This is limited till experimentations but in a production application, you should switch to use SqliteSaver or PostgresSaver and connect to your own DB to store the chat memory of graph.

In [ ]:
# Building the Agent with Memory

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

tool = TavilySearchResults(max_results=5)
tools = [tool]
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile(checkpointer=memory)

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
config = {"configurable": {"thread_id": "User_1"}}

user_input = "Hi, my name is Himanshu"

events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()

In [ ]:
user_input = "Do you know my name?"

events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()

### What would happen if I change my config

In [ ]:
config = {"configurable": {"thread_id": "User_2"}}

user_input = "Do you know my name?"

events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()

As expected, as the thread id has changed, it doesn't have access to memory of other thread

In [ ]:
config = {"configurable": {"thread_id": "User_1"}}

user_input = "Do you know my name?"

events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()

In [ ]:
config = {"configurable": {"thread_id": "User_2"}}

user_input = "Hi my name is Ajit and I live in Indore"

events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()

In [ ]:
config = {"configurable": {"thread_id": "User_1"}}

user_input = "Do you know where I live ?"

events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()

## Snapshot of current state

In [ ]:
snapshot = graph.get_state(config)
snapshot